<a href="https://colab.research.google.com/github/cn8972/Echo-Bot/blob/main/Degree%20Match%20for%20Cybersecurity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# UMGC Undergraduate Cybersecurity Program Recommender
# Google Colab + Gradio + scikit-learn
# Hybrid expert system + ML on synthetic data
# ============================================

!pip -q install gradio scikit-learn pandas numpy

import numpy as np
import pandas as pd
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

# --------------------------------------------------
# 1. Program definitions
# --------------------------------------------------

PROGRAMS = {
    "Cybersecurity Technology": {
        "summary": (
            "Best aligned for students who want broad hands-on technical preparation in "
            "networking, systems, ethical hacking, troubleshooting, cloud, and operational cybersecurity."
        ),
        "best_for": [
            "students who want practical technical breadth",
            "students interested in networks, systems, and cyber defense",
            "students who like troubleshooting and hands-on labs",
        ],
        "careers": [
            "cybersecurity analyst",
            "security engineer",
            "systems administrator",
            "network/security administrator",
            "cloud/security support roles"
        ]
    },
    "Cyber Operations": {
        "summary": (
            "Best aligned for students who enjoy programming, offensive and defensive operations, "
            "detection, alert triage, automation, scripting, and highly technical cyber problem-solving."
        ),
        "best_for": [
            "students who enjoy coding and technical depth",
            "students interested in blue team/red team activity",
            "students who like fast-paced operational problem-solving",
        ],
        "careers": [
            "security operations analyst",
            "threat detection analyst",
            "incident responder",
            "cyber operator",
            "technical security engineer"
        ]
    },
    "Cybersecurity Management & Policy": {
        "summary": (
            "Best aligned for students interested in cyber leadership, governance, risk, compliance, "
            "policy, communication, strategy, and protecting organizations through management decisions."
        ),
        "best_for": [
            "students strong in communication and leadership",
            "students interested in governance, law, policy, and risk",
            "students who prefer strategic over deeply technical work",
        ],
        "careers": [
            "risk analyst",
            "cyber policy analyst",
            "compliance analyst",
            "security program coordinator",
            "governance/risk/support roles"
        ]
    }
}

PROGRAM_ORDER = list(PROGRAMS.keys())

# --------------------------------------------------
# 2. Feature schema
# --------------------------------------------------
# Scale convention:
# 1 = low / dislike / weak
# 5 = high / strong / prefer

FEATURE_NAMES = [
    "math_confidence",
    "problem_solving",
    "programming_interest",
    "networking_interest",
    "hands_on_lab_interest",
    "leadership_interest",
    "writing_policy_interest",
    "risk_governance_interest",
    "offensive_security_interest",
    "defensive_security_interest",
    "cloud_interest",
    "communication_strength",
    "detail_orientation",
    "fast_paced_env_preference",
    "career_goal_code_ops",       # encoded from dropdown
    "career_goal_tech_defense",   # encoded from dropdown
    "career_goal_management",     # encoded from dropdown
    "personality_analytical",
    "personality_people_oriented",
    "personality_investigative",
    "personality_structured"
]

# --------------------------------------------------
# 3. Expert scoring rules
# --------------------------------------------------

def compute_rule_scores(features: dict):
    """
    Compute expert rule-based scores for each UMGC program.
    """
    scores = {
        "Cybersecurity Technology": 0.0,
        "Cyber Operations": 0.0,
        "Cybersecurity Management & Policy": 0.0
    }

    # Cybersecurity Technology
    scores["Cybersecurity Technology"] += (
        1.3 * features["networking_interest"] +
        1.2 * features["hands_on_lab_interest"] +
        1.1 * features["defensive_security_interest"] +
        1.0 * features["cloud_interest"] +
        1.0 * features["problem_solving"] +
        0.9 * features["detail_orientation"] +
        0.8 * features["math_confidence"] +
        0.7 * features["programming_interest"] +
        1.5 * features["career_goal_tech_defense"] +
        0.7 * features["personality_investigative"] +
        0.6 * features["personality_structured"]
    )

    # Cyber Operations
    scores["Cyber Operations"] += (
        1.5 * features["programming_interest"] +
        1.2 * features["math_confidence"] +
        1.3 * features["problem_solving"] +
        1.2 * features["offensive_security_interest"] +
        1.1 * features["defensive_security_interest"] +
        0.8 * features["networking_interest"] +
        1.0 * features["hands_on_lab_interest"] +
        0.9 * features["fast_paced_env_preference"] +
        1.5 * features["career_goal_code_ops"] +
        0.9 * features["personality_analytical"] +
        0.9 * features["personality_investigative"]
    )

    # Cybersecurity Management & Policy
    scores["Cybersecurity Management & Policy"] += (
        1.4 * features["leadership_interest"] +
        1.4 * features["writing_policy_interest"] +
        1.5 * features["risk_governance_interest"] +
        1.2 * features["communication_strength"] +
        1.0 * features["detail_orientation"] +
        0.8 * features["problem_solving"] +
        1.5 * features["career_goal_management"] +
        0.8 * features["personality_people_oriented"] +
        0.8 * features["personality_structured"] +
        0.7 * features["personality_analytical"]
    )

    return scores


def choose_program_by_rules(features: dict):
    scores = compute_rule_scores(features)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    best_program = ranked[0][0]
    return best_program, scores, ranked


# --------------------------------------------------
# 4. Synthetic data generation for ML training
# --------------------------------------------------

def random_feature_profile():
    return {
        "math_confidence": np.random.randint(1, 6),
        "problem_solving": np.random.randint(1, 6),
        "programming_interest": np.random.randint(1, 6),
        "networking_interest": np.random.randint(1, 6),
        "hands_on_lab_interest": np.random.randint(1, 6),
        "leadership_interest": np.random.randint(1, 6),
        "writing_policy_interest": np.random.randint(1, 6),
        "risk_governance_interest": np.random.randint(1, 6),
        "offensive_security_interest": np.random.randint(1, 6),
        "defensive_security_interest": np.random.randint(1, 6),
        "cloud_interest": np.random.randint(1, 6),
        "communication_strength": np.random.randint(1, 6),
        "detail_orientation": np.random.randint(1, 6),
        "fast_paced_env_preference": np.random.randint(1, 6),
        "career_goal_code_ops": 0,
        "career_goal_tech_defense": 0,
        "career_goal_management": 0,
        "personality_analytical": np.random.randint(1, 6),
        "personality_people_oriented": np.random.randint(1, 6),
        "personality_investigative": np.random.randint(1, 6),
        "personality_structured": np.random.randint(1, 6),
    }


def assign_random_career_goal(profile):
    choice = np.random.choice(["ops", "tech", "mgmt"], p=[0.33, 0.34, 0.33])
    if choice == "ops":
        profile["career_goal_code_ops"] = 1
    elif choice == "tech":
        profile["career_goal_tech_defense"] = 1
    else:
        profile["career_goal_management"] = 1
    return profile


def generate_synthetic_dataset(n=5000, random_state=42):
    np.random.seed(random_state)
    rows = []

    for _ in range(n):
        f = random_feature_profile()
        f = assign_random_career_goal(f)

        # Add mild structure to make synthetic profiles more realistic
        # If programming is high, analytical and math often trend upward slightly
        if f["programming_interest"] >= 4:
            f["personality_analytical"] = min(5, f["personality_analytical"] + np.random.randint(0, 2))
            f["math_confidence"] = min(5, f["math_confidence"] + np.random.randint(0, 2))

        # If leadership/policy is high, communication often trends upward
        if f["leadership_interest"] >= 4 or f["writing_policy_interest"] >= 4:
            f["communication_strength"] = min(5, f["communication_strength"] + np.random.randint(0, 2))

        # If networking/lab are high, defensive/cloud interest often trend upward
        if f["networking_interest"] >= 4 or f["hands_on_lab_interest"] >= 4:
            f["defensive_security_interest"] = min(5, f["defensive_security_interest"] + np.random.randint(0, 2))
            f["cloud_interest"] = min(5, f["cloud_interest"] + np.random.randint(0, 2))

        label, _, _ = choose_program_by_rules(f)

        row = [f[name] for name in FEATURE_NAMES] + [label]
        rows.append(row)

    columns = FEATURE_NAMES + ["label"]
    return pd.DataFrame(rows, columns=columns)


# --------------------------------------------------
# 5. Train ML model
# --------------------------------------------------

df = generate_synthetic_dataset(n=6000, random_state=7)

X = df[FEATURE_NAMES]
y = df["label"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Model trained.\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


# --------------------------------------------------
# 6. Input processing
# --------------------------------------------------

CAREER_MAP = {
    "Coding / security operations / threat response": ("career_goal_code_ops",),
    "Hands-on technical cybersecurity / networking / cloud": ("career_goal_tech_defense",),
    "Leadership / risk / compliance / cyber policy": ("career_goal_management",),
}

def build_feature_dict(
    math_confidence,
    problem_solving,
    programming_interest,
    networking_interest,
    hands_on_lab_interest,
    leadership_interest,
    writing_policy_interest,
    risk_governance_interest,
    offensive_security_interest,
    defensive_security_interest,
    cloud_interest,
    communication_strength,
    detail_orientation,
    fast_paced_env_preference,
    career_goal,
    personality_analytical,
    personality_people_oriented,
    personality_investigative,
    personality_structured
):
    features = {
        "math_confidence": int(math_confidence),
        "problem_solving": int(problem_solving),
        "programming_interest": int(programming_interest),
        "networking_interest": int(networking_interest),
        "hands_on_lab_interest": int(hands_on_lab_interest),
        "leadership_interest": int(leadership_interest),
        "writing_policy_interest": int(writing_policy_interest),
        "risk_governance_interest": int(risk_governance_interest),
        "offensive_security_interest": int(offensive_security_interest),
        "defensive_security_interest": int(defensive_security_interest),
        "cloud_interest": int(cloud_interest),
        "communication_strength": int(communication_strength),
        "detail_orientation": int(detail_orientation),
        "fast_paced_env_preference": int(fast_paced_env_preference),
        "career_goal_code_ops": 0,
        "career_goal_tech_defense": 0,
        "career_goal_management": 0,
        "personality_analytical": int(personality_analytical),
        "personality_people_oriented": int(personality_people_oriented),
        "personality_investigative": int(personality_investigative),
        "personality_structured": int(personality_structured),
    }

    for key in CAREER_MAP.get(career_goal, ()):
        features[key] = 1

    return features


# --------------------------------------------------
# 7. Recommendation engine
# --------------------------------------------------

def recommend_program(
    math_confidence,
    problem_solving,
    programming_interest,
    networking_interest,
    hands_on_lab_interest,
    leadership_interest,
    writing_policy_interest,
    risk_governance_interest,
    offensive_security_interest,
    defensive_security_interest,
    cloud_interest,
    communication_strength,
    detail_orientation,
    fast_paced_env_preference,
    career_goal,
    personality_analytical,
    personality_people_oriented,
    personality_investigative,
    personality_structured
):
    features = build_feature_dict(
        math_confidence,
        problem_solving,
        programming_interest,
        networking_interest,
        hands_on_lab_interest,
        leadership_interest,
        writing_policy_interest,
        risk_governance_interest,
        offensive_security_interest,
        defensive_security_interest,
        cloud_interest,
        communication_strength,
        detail_orientation,
        fast_paced_env_preference,
        career_goal,
        personality_analytical,
        personality_people_oriented,
        personality_investigative,
        personality_structured
    )

    # Rule-based result
    rule_program, rule_scores, ranked_rules = choose_program_by_rules(features)

    # ML result
    x_user = pd.DataFrame([[features[name] for name in FEATURE_NAMES]], columns=FEATURE_NAMES)
    ml_probs = model.predict_proba(x_user)[0]
    ml_ranked_idx = np.argsort(ml_probs)[::-1]
    ml_top_program = label_encoder.inverse_transform([ml_ranked_idx[0]])[0]

    # Blend rule and ML scores
    # Normalize rule scores
    raw_rule = np.array([rule_scores[p] for p in PROGRAM_ORDER], dtype=float)
    rule_norm = raw_rule / raw_rule.sum()

    # Align ML probs to PROGRAM_ORDER
    ml_prob_map = {
        label_encoder.inverse_transform([i])[0]: ml_probs[i]
        for i in range(len(ml_probs))
    }
    ml_ordered = np.array([ml_prob_map[p] for p in PROGRAM_ORDER], dtype=float)

    blended = 0.55 * rule_norm + 0.45 * ml_ordered
    best_idx = int(np.argmax(blended))
    recommended_program = PROGRAM_ORDER[best_idx]

    # Explanations
    strengths = []
    if features["programming_interest"] >= 4:
        strengths.append("strong interest in programming")
    if features["networking_interest"] >= 4:
        strengths.append("strong interest in networking")
    if features["hands_on_lab_interest"] >= 4:
        strengths.append("preference for hands-on labs")
    if features["offensive_security_interest"] >= 4:
        strengths.append("interest in offensive security")
    if features["defensive_security_interest"] >= 4:
        strengths.append("interest in defensive security")
    if features["risk_governance_interest"] >= 4:
        strengths.append("interest in governance and risk")
    if features["writing_policy_interest"] >= 4:
        strengths.append("interest in writing, policy, or compliance")
    if features["leadership_interest"] >= 4:
        strengths.append("leadership orientation")
    if features["communication_strength"] >= 4:
        strengths.append("strong communication preference")
    if features["problem_solving"] >= 4:
        strengths.append("high confidence in problem-solving")
    if features["math_confidence"] >= 4:
        strengths.append("comfort with math/technical reasoning")

    if not strengths:
        strengths.append("balanced interests across multiple areas")

    top3 = np.argsort(blended)[::-1][:3]
    ranking_lines = []
    for idx in top3:
        prog = PROGRAM_ORDER[idx]
        ranking_lines.append(f"- {prog}: {blended[idx]*100:.1f}% fit")

    rationale = PROGRAMS[recommended_program]["summary"]
    best_for = "\n".join([f"- {item}" for item in PROGRAMS[recommended_program]["best_for"]])
    careers = "\n".join([f"- {item}" for item in PROGRAMS[recommended_program]["careers"]])

    result = f"""
# Recommended UMGC Program
**{recommended_program}**

## Why this program fits
{rationale}

## Your strongest matching signals
- """ + "\n- ".join(strengths) + f"""

## Top program ranking
{chr(10).join(ranking_lines)}

## This program is often a strong fit for
{best_for}

## Example career directions
{careers}

## Advisory note
This is a prototype recommendation based on a hybrid expert-rule and machine-learning model.
It should be used as a starting point for discussion with an academic advisor, not as a final enrollment decision.
"""

    # Return a simple dataframe for transparency
    detail_df = pd.DataFrame({
        "Program": PROGRAM_ORDER,
        "Rule Score (normalized)": np.round(rule_norm, 4),
        "ML Probability": np.round(ml_ordered, 4),
        "Blended Fit Score": np.round(blended, 4)
    }).sort_values("Blended Fit Score", ascending=False)

    return result, detail_df


# --------------------------------------------------
# 8. Build Gradio UI
# --------------------------------------------------

with gr.Blocks(title="UMGC Cybersecurity Program Recommender") as demo:
    gr.Markdown(
        """
        # UMGC Undergraduate Cybersecurity Program Recommender
        This prototype asks students about interests, strengths, and preferences, then recommends the best-fit UMGC undergraduate cybersecurity program.
        """
    )

    with gr.Row():
        with gr.Column():
            math_confidence = gr.Slider(1, 5, value=3, step=1, label="How confident are you in math and technical reasoning?")
            problem_solving = gr.Slider(1, 5, value=4, step=1, label="How strong are your problem-solving skills?")
            programming_interest = gr.Slider(1, 5, value=3, step=1, label="How interested are you in programming/coding?")
            networking_interest = gr.Slider(1, 5, value=3, step=1, label="How interested are you in networking and systems?")
            hands_on_lab_interest = gr.Slider(1, 5, value=4, step=1, label="How much do you enjoy hands-on labs and technical exercises?")
            leadership_interest = gr.Slider(1, 5, value=3, step=1, label="How interested are you in leadership and managing security efforts?")
            writing_policy_interest = gr.Slider(1, 5, value=2, step=1, label="How interested are you in writing, policy, compliance, or governance?")
            risk_governance_interest = gr.Slider(1, 5, value=2, step=1, label="How interested are you in risk assessment and governance?")
            offensive_security_interest = gr.Slider(1, 5, value=3, step=1, label="How interested are you in offensive security or ethical hacking?")
            defensive_security_interest = gr.Slider(1, 5, value=4, step=1, label="How interested are you in defensive security and protection?")

        with gr.Column():
            cloud_interest = gr.Slider(1, 5, value=3, step=1, label="How interested are you in cloud technologies?")
            communication_strength = gr.Slider(1, 5, value=3, step=1, label="How strong are your communication skills?")
            detail_orientation = gr.Slider(1, 5, value=4, step=1, label="How detail-oriented are you?")
            fast_paced_env_preference = gr.Slider(1, 5, value=3, step=1, label="How much do you enjoy fast-paced operational environments?")
            career_goal = gr.Radio(
                choices=[
                    "Coding / security operations / threat response",
                    "Hands-on technical cybersecurity / networking / cloud",
                    "Leadership / risk / compliance / cyber policy"
                ],
                value="Hands-on technical cybersecurity / networking / cloud",
                label="Which career path sounds most appealing right now?"
            )
            personality_analytical = gr.Slider(1, 5, value=4, step=1, label="How analytical are you?")
            personality_people_oriented = gr.Slider(1, 5, value=3, step=1, label="How people-oriented are you?")
            personality_investigative = gr.Slider(1, 5, value=4, step=1, label="How investigative/curious are you?")
            personality_structured = gr.Slider(1, 5, value=4, step=1, label="How much do you prefer structure, process, and clear rules?")

    submit_btn = gr.Button("Recommend My Program")

    result_text = gr.Markdown()
    result_table = gr.Dataframe(label="Model Details", interactive=False)

    submit_btn.click(
        fn=recommend_program,
        inputs=[
            math_confidence,
            problem_solving,
            programming_interest,
            networking_interest,
            hands_on_lab_interest,
            leadership_interest,
            writing_policy_interest,
            risk_governance_interest,
            offensive_security_interest,
            defensive_security_interest,
            cloud_interest,
            communication_strength,
            detail_orientation,
            fast_paced_env_preference,
            career_goal,
            personality_analytical,
            personality_people_oriented,
            personality_investigative,
            personality_structured
        ],
        outputs=[result_text, result_table]
    )

demo.launch(share=True, debug=True)

Model trained.

                                   precision    recall  f1-score   support

                 Cyber Operations       0.78      1.00      0.88       785
Cybersecurity Management & Policy       0.95      0.56      0.70       326
         Cybersecurity Technology       1.00      0.08      0.15        89

                         accuracy                           0.81      1200
                        macro avg       0.91      0.55      0.58      1200
                     weighted avg       0.84      0.81      0.78      1200

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c37375badf3cd6ab9c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
